# 🕸️ Lesson 4.2 — Building a Neural Network: Layers and Depth

**AI Crash Course for Absolute Beginners**

Stack neurons into layers and you get a neural network that can learn any pattern. In this notebook:
- Build a 2-layer network from scratch (pure NumPy)
- Implement forward pass, loss, and backpropagation
- Solve XOR (which the single neuron could not)
- Build a full network with PyTorch and train on real data

> Run each cell with **Shift + Enter**.

In [ ]:
# Install libraries (already in Colab — this is a safety net)
!pip install numpy matplotlib torch torchvision --quiet

# numpy — fast maths and arrays
import numpy as np

# matplotlib — drawing charts
import matplotlib.pyplot as plt

---
## Part 1 — A 2-Layer Network from Scratch

In [ ]:
class TwoLayerNetwork:
    """
    A simple 2-layer neural network built from scratch using only NumPy.
    Architecture: 2 inputs → 4 hidden neurons (ReLU) → 1 output (sigmoid)
    This shows exactly what PyTorch does behind the scenes.
    """

    def __init__(self, input_size, hidden_size, lr=0.1):
        # Xavier initialisation: smarter than random — prevents gradients from vanishing or exploding
        # The scale is based on the number of inputs to each layer
        scale1 = np.sqrt(2.0 / input_size)
        scale2 = np.sqrt(2.0 / hidden_size)

        # W1: weight matrix connecting inputs to hidden layer (shape: input_size x hidden_size)
        self.W1 = np.random.randn(input_size, hidden_size) * scale1
        self.b1 = np.zeros((1, hidden_size))   # one bias per hidden neuron, start at 0

        # W2: weight matrix connecting hidden layer to output (shape: hidden_size x 1)
        self.W2 = np.random.randn(hidden_size, 1) * scale2
        self.b2 = np.zeros((1, 1))             # one bias for the output neuron

        self.lr = lr   # learning rate: how large each weight update step is

    # Activation functions
    def relu(self, z):        return np.maximum(0, z)          # ReLU: output z if positive, else 0
    def relu_grad(self, z):   return (z > 0).astype(float)     # ReLU derivative: 1 if z>0, else 0
    def sigmoid(self, z):     return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # clip avoids overflow

    def forward(self, X):
        """Forward pass: data flows from inputs to output, producing a prediction."""
        self.Z1 = X @ self.W1 + self.b1      # hidden layer: weighted sum (@ = matrix multiply)
        self.A1 = self.relu(self.Z1)          # apply ReLU activation to hidden layer
        self.Z2 = self.A1 @ self.W2 + self.b2 # output layer: weighted sum
        self.A2 = self.sigmoid(self.Z2)        # apply sigmoid to get a probability (0 to 1)
        return self.A2

    def backward(self, X, y):
        """Backward pass: calculate how much each weight contributed to the error,
        then nudge weights in the direction that reduces the error (gradient descent).
        """
        m = len(y)   # number of training samples

        # --- Output layer gradients ---
        # How wrong was the output? (derivative of cross-entropy loss + sigmoid combined)
        dZ2 = self.A2 - y.reshape(-1, 1)
        dW2 = self.A1.T @ dZ2 / m      # how much should W2 change? (T = transpose)
        db2 = dZ2.mean(axis=0, keepdims=True)   # how much should b2 change?

        # --- Hidden layer gradients (chain rule: multiply gradients backwards through layers) ---
        dA1 = dZ2 @ self.W2.T          # gradient flowing back through W2
        dZ1 = dA1 * self.relu_grad(self.Z1)  # gradient through ReLU (zero where Z1 was negative)
        dW1 = X.T @ dZ1 / m            # how much should W1 change?
        db1 = dZ1.mean(axis=0, keepdims=True)   # how much should b1 change?

        # --- Update weights: subtract gradient * learning rate ---
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def fit(self, X, y, epochs=5000):
        losses = []
        for epoch in range(epochs):
            y_hat = self.forward(X).flatten()   # run forward pass, flatten to 1D array
            # Cross-entropy loss: lower = better predictions
            loss = -np.mean(y * np.log(y_hat + 1e-9) + (1 - y) * np.log(1 - y_hat + 1e-9))
            self.backward(X, y)    # adjust weights
            losses.append(loss)
        return losses

print("TwoLayerNetwork class defined.")

In [ ]:
# Solve XOR
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0,1,1,0], dtype=float)

np.random.seed(42)
net = TwoLayerNetwork(input_size=2, hidden_size=4, lr=0.5)
losses = net.fit(X_xor, y_xor, epochs=5000)

preds = (net.forward(X_xor).flatten() > 0.5).astype(int)
print("XOR predictions after training:")
for xi, yi, pred in zip(X_xor, y_xor, preds):
    ok = "OK" if pred == yi else "WRONG"
    print(f"  {int(xi[0])} XOR {int(xi[1])} = {pred}  (true: {int(yi)})  {ok}")

plt.figure(figsize=(6, 3))
plt.plot(losses, color="#c8401a")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Neural Network Solves XOR")
plt.tight_layout()
plt.show()

---
## Part 2 — PyTorch: The Professional Way

In [ ]:
# PyTorch — the most popular deep learning library (used by Meta, OpenAI, and most researchers)
import torch

# torch.nn — contains all the building blocks for neural networks (layers, loss functions)
import torch.nn as nn

# torch.optim — contains optimisers (algorithms that update weights during training)
import torch.optim as optim

# DataLoader — feeds data to the model in small batches during training
# TensorDataset — wraps data and labels together into one object
from torch.utils.data import DataLoader, TensorDataset

# scikit-learn tools for data preparation
from sklearn.datasets import make_classification   # generates a synthetic classification dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"PyTorch version: {torch.__version__}")

# Check if a GPU is available — PyTorch will use it automatically if so (much faster training)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}  {'(GPU — nice!)' if device == 'cuda' else '(CPU — still fine for this notebook)'}")

In [ ]:
# Generate a synthetic dataset — 1000 samples, 20 features, 10 of which are actually useful
X_raw, y_raw = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=42)

# Split: 80% training, 20% testing
X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# Scale features so all inputs are on the same scale (mean=0, std=1)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)   # learn scaling from training data, then apply it
X_te = scaler.transform(X_te)       # apply the SAME scaling to test data (don't refit!)

# --- Convert NumPy arrays to PyTorch tensors ---
# PyTorch works with tensors, not NumPy arrays
# FloatTensor means 32-bit floating point numbers (the default for neural networks)
# .to(device) sends the data to GPU if available, otherwise keeps it on CPU
X_tr_t = torch.FloatTensor(X_tr).to(device)
y_tr_t = torch.FloatTensor(y_tr).to(device)
X_te_t = torch.FloatTensor(X_te).to(device)
y_te_t = torch.FloatTensor(y_te).to(device)

# DataLoader batches the data — instead of training on all 800 samples at once,
# we feed the model 32 samples at a time (batch_size=32)
# shuffle=True randomises the order each epoch (helps training)
train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)
print(f"Train batches: {len(train_loader)}  (800 samples ÷ 32 per batch)")

In [ ]:
class FeedForwardNet(nn.Module):
    """A fully connected neural network built with PyTorch.
    nn.Module is the base class for all PyTorch neural networks — we always inherit from it.
    """

    def __init__(self, input_size, hidden_sizes, output_size):
        super().__init__()   # always call this first when inheriting from nn.Module

        layers = []          # we'll build a list of layers, then wrap them in Sequential
        prev = input_size    # track the input size of the current layer

        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))   # fully connected layer: prev inputs → h neurons
            layers.append(nn.ReLU())             # ReLU activation after each hidden layer
            layers.append(nn.Dropout(0.2))       # Dropout: randomly turns off 20% of neurons during training
            prev = h                             # next layer's input size = this layer's output size

        layers.append(nn.Linear(prev, output_size))  # final output layer (no activation — loss handles it)

        # nn.Sequential chains all layers together: input flows through each one in order
        self.net = nn.Sequential(*layers)   # *layers unpacks the list into separate arguments

    def forward(self, x):
        # squeeze() removes the extra dimension — turns shape [batch, 1] into [batch]
        return self.net(x).squeeze()

# Create the network: 20 inputs → 64 neurons → 32 neurons → 1 output
model = FeedForwardNet(20, [64, 32], 1).to(device)
print(model)

# Count how many weights the model has in total
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

In [ ]:
# Loss function: BCEWithLogitsLoss = Binary Cross-Entropy
# More numerically stable than applying sigmoid first then BCELoss
criterion = nn.BCEWithLogitsLoss()

# Adam optimiser: a smarter version of gradient descent that adapts the learning rate automatically
# lr=1e-3 means learning rate of 0.001
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_losses, val_losses = [], []
EPOCHS = 50

for epoch in range(EPOCHS):

    # === TRAINING PHASE ===
    model.train()   # puts model in training mode (enables Dropout)
    batch_losses = []

    for X_batch, y_batch in train_loader:   # loop through mini-batches of 32 samples
        optimizer.zero_grad()               # clear gradients from the previous batch (important!)
        logits = model(X_batch)             # forward pass: get raw predictions
        loss   = criterion(logits, y_batch) # calculate how wrong the predictions are
        loss.backward()                     # backpropagation: calculate gradients
        optimizer.step()                    # update weights using the gradients
        batch_losses.append(loss.item())    # .item() converts tensor to a Python number

    # === VALIDATION PHASE ===
    model.eval()   # puts model in evaluation mode (disables Dropout)
    with torch.no_grad():   # no_grad: don't calculate gradients (saves memory, speeds up)
        val_loss = criterion(model(X_te_t), y_te_t).item()

    train_losses.append(np.mean(batch_losses))
    val_losses.append(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}")

In [ ]:
# Plot training curves
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train loss", color="#1a6bc8")
plt.plot(val_losses,   label="Val loss",   color="#c8401a")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Final accuracy
model.eval()
with torch.no_grad():
    preds = (torch.sigmoid(model(X_te_t)) > 0.5).float()
    acc   = (preds == y_te_t).float().mean().item()
print(f"Test accuracy: {acc:.3f}")

---
## Challenge Exercises

1. **Width experiment**: Change hidden sizes from `[64, 32]` to `[256, 128, 64]`. Does accuracy improve? Does training slow?
2. **Dropout tuning**: Change `Dropout(0.2)` to `Dropout(0.5)`. How does the train/val gap change?
3. **Learning rate**: Try `lr=0.01` and `lr=0.0001`. Compare loss curves.

---
**Next lesson:** [Lesson 4.3 — Convolutional Neural Networks](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-4.3-cnn.ipynb)